In [3]:
from pdf_analysis import PdfDetector
import pandas as pd
import numpy as np
import os, json
import fitz
from pathlib import Path
import time

from collections import Counter

TEXT OR SCANNED

In [ ]:
#new pdf
import pandas as pd
import numpy as np
import os, json

folder_pdf = r"C:\Users\kaustubh.keny\Documents\Quarterly Results 2026 Q1"
def page_text_or_scanned(page):

    text = page.get_text("text").strip()
    page_rect = page.rect
    page_area = page_rect.width * page_rect.height

    if page_area <= 0:
        return "scanned"

    image_area = 0
    for img in page.get_images(full=True):
        try:
            xref = img[0]
            for rect in page.get_image_rects(xref):
                clipped = rect & page_rect
                if clipped.is_empty:
                    continue

                rect_area = clipped.width * clipped.height
                # Ignore small logos
                if rect_area > page_area * 0.05:
                    image_area += rect_area

        except Exception:
            continue

    image_coverage = min(image_area / page_area, 1.0)
    blocks = page.get_text("blocks")
    text_blocks = [
        block for block in blocks if len(block) >= 5 and str(block[4]).strip()
    ]

    num_text_blocks = len(text_blocks)

    # Strong text page
    if len(text) > 100 and num_text_blocks >= 3 and image_coverage < 0.8:
        return "text"
    # Strong scanned page
    if image_coverage > 0.8 and len(text) < 100:
        return "scanned"
    # OCR scanned page
    if image_coverage > 0.9 and num_text_blocks <= 2:
        return "scanned"
    return "text" if len(text) > 100 else "scanned"


all_data = []

for file in os.listdir(folder_pdf):

    file_path = os.path.join(folder_pdf, file)

    stem = Path(file_path).name
    print(f"\nProcessing: {stem}")

    doc = fitz.open(file_path)

    for idx, page in enumerate(doc):
        
        res = page_text_or_scanned(page)

        # print("Result:", res)

        all_data.append({
            "pdf_name":stem,
            "page_n":idx +1,
            "type":res
        })
        
    doc.close()


fpath = Path(folder_pdf)

# Overwrite original file
df = pd.DataFrame(all_data)
df.to_excel(f"{fpath.stem}_TYPE.xlsx" ,index=False)



Processing: Birlasoft Ltd-Sep-25.pdf

Processing: Cyient Ltd-Q4-Mar-2026.pdf

Processing: Just Dial Ltd-Q4-Mar-26.pdf

Processing: Persistent Systems Ltd-Jun-26.pdf

Processing: Tanla Platforms Ltd-Jun-26.pdf

Processing: Tata Consulatancy Serivces Ltd-Dec-25.pdf

Processing: Tata Elxsi Ltd-Sep-25.pdf

Processing: Wherrelz IT Solutions Ltd-Q4-Mar-2026.pdf


In [ ]:
#saved pdf
path =r"PDF_PAGE.xlsx"
folder_pdf = r"C:\Users\kaustubh.keny\Downloads\QUARTERLY_REPORTS"
def page_text_or_scanned(page):

    text = page.get_text("text").strip()
    page_rect = page.rect
    page_area = page_rect.width * page_rect.height

    if page_area <= 0:
        return "scanned"

    image_area = 0
    for img in page.get_images(full=True):
        try:
            xref = img[0]
            for rect in page.get_image_rects(xref):
                clipped = rect & page_rect
                if clipped.is_empty:
                    continue

                rect_area = clipped.width * clipped.height
                # Ignore small logos
                if rect_area > page_area * 0.05:
                    image_area += rect_area

        except Exception:
            continue

    image_coverage = min(image_area / page_area, 1.0)
    blocks = page.get_text("blocks")
    text_blocks = [
        block for block in blocks if len(block) >= 5 and str(block[4]).strip()
    ]

    num_text_blocks = len(text_blocks)

    # Strong text page
    if len(text) > 100 and num_text_blocks >= 3 and image_coverage < 0.8:
        return "text"
    # Strong scanned page
    if image_coverage > 0.8 and len(text) < 100:
        return "scanned"
    # OCR scanned page
    if image_coverage > 0.9 and num_text_blocks <= 2:
        return "scanned"
    return "text" if len(text) > 100 else "scanned"

df = pd.read_excel(path, sheet_name="2026")
df.head(4)

for file in os.listdir(folder_pdf):

    file_path = os.path.join(folder_pdf, file)

    stem = Path(file_path).name
    print(f"\nProcessing: {stem}")

    doc = fitz.open(file_path)

    mask = df["pdf_name"].str.contains(stem, na=False)

    # print("Matched rows:", mask.sum())

    for idx, row in df.loc[mask].iterrows():

        # print("Row:", idx)

        page_n = int(row["page_number"]) - 1

        res = page_text_or_scanned(doc[page_n])

        print("Result:", res)

        df.at[idx, "page_type"] = res

    doc.close()

print(df.head())

# Overwrite original file
df.to_excel(path, sheet_name="2026",index=False)

print(f"Saved updates to: {path}")

In [2]:
# Overwrite original file
df.to_excel(path, sheet_name="2026",index=False)

print(f"Saved updates to: {path}")

Saved updates to: PDF_PAGE.xlsx


WORD CLOUD

In [ ]:
import fitz  # PyMuPDF
import re
import json
from collections import Counter
from wordcloud import WordCloud
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import landscape, letter
from reportlab.lib.utils import ImageReader
from io import BytesIO
import matplotlib.pyplot as plt

def pdf_side_by_side_inmemory(pdf_path,till = 200,
                              output_pdf="compare_wordclouds1.pdf",
                              output_json="pagewise_word_freq1.json"):
    doc = fitz.open(pdf_path)
    pagewise_freq = {}
    if doc.page_count< till:
        till = doc.page_count

    for i, page in enumerate(doc, start=1):
        if i > till:
            break
        text = page.get_text()
        words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
        freq = Counter(words)
        pagewise_freq[f"page_{i}"] = dict(freq)

    
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(pagewise_freq, f, indent=4)


    c = canvas.Canvas(output_pdf, pagesize=landscape(letter))
    width, height = landscape(letter)

    for i, page in enumerate(doc, start=1):
        if i > till:
            break
        pix = page.get_pixmap(matrix=fitz.Matrix(0.7, 0.7))
        orig_bytes = BytesIO(pix.tobytes("png"))
        orig_img = ImageReader(orig_bytes)

        freq = pagewise_freq[f"page_{i}"]
        if freq:
            wc = WordCloud(width=600, height=600, background_color="white")
            wc.generate_from_frequencies(freq)

            buf = BytesIO()
            plt.figure(figsize=(6,6))
            plt.imshow(wc, interpolation="bilinear")
            plt.axis("off")
            plt.savefig(buf, format="png", bbox_inches="tight")
            plt.close()
            buf.seek(0)
            wc_img = ImageReader(buf)

            # Place original on left, word cloud on right
            c.drawImage(orig_img, 20, 50, width=350, height=500)
            c.drawImage(wc_img, 420, 50, width=350, height=500)
            c.drawString(width/2 - 50, height - 30, f"Page {i} Comparison")
            c.showPage()

    c.save()
    print(f"Combined PDF saved as {output_pdf}")
    print(f"Page-wise JSON saved as {output_json}")


path =r"FIRST10.pdf"
pdf_side_by_side_inmemory(path)


Combined PDF saved as compare_wordclouds1.pdf
Page-wise JSON saved as pagewise_word_freq1.json


FIRST 10 PAGES

In [9]:
import fitz
input_folder = r"C:\Users\kaustubh.keny\Downloads\FINANCE"
pdf_files = sorted(Path(input_folder).glob("*.pdf"))
merged_pdf = fitz.open()

for pdf in pdf_files:
    src = fitz.open(pdf)
    merged_pdf.insert_pdf(src)
    src.close()

merged_pdf.save("MERGED_FINANCE.pdf")
merged_pdf.close()

In [ ]:
#CUT PDF + MERGE
import fitz
from pathlib import Path

input_folder = r"C:\Users\kaustubh.keny\Projects\INPUTS\ANNUAL_REPORTS\ANNUAL_REPORTS_2026"
output_pdf = "FIRST2.pdf"

merged_doc = fitz.open()

for pdf_file in sorted(Path(input_folder).glob("*.pdf")):
    try:
        src = fitz.open(pdf_file)
        #if src.page_count > 0:
        merged_doc.insert_pdf(src, from_page=0, to_page=0)

        src.close()
        print(f"Added {pdf_file.name}")

    except Exception as e:
        print(f"Error processing {pdf_file.name}: {e}")

print(f"Total pages: {merged_doc.page_count}")

merged_doc.save(output_pdf)
merged_doc.close()

print(f"Saved: {output_pdf}")

In [ ]:
import re
import fitz
from pathlib import Path

input_folder = r"C:\Users\kaustubh.keny\Projects\INPUTS\ANNUAL_REPORTS\ANNUAL_REPORTS_2026"
output_pdf = "INDEX261.pdf"

terms = [
    r"CORPORATE\s+OVERVIEW",
    r"STATUTORY\s+REPORTS?",
    r"FINANCIAL\s+(?:STATEMENTS?|REPORTS?)",
    r"\bCONTENTS\b",
    r"CORPORATE\s+GOVERNANCE\s+REPORT",
    r"BUSINESS\s+RESPONSIBILITY",
    r"SUSTAINABILITY\s+REPORT",
    r"STANDALONE(?:\s+FINANCIAL)?",
    r"CONSOLIDATED(?:\s+FINANCIAL)?",
]

pattern = re.compile("|".join(terms), re.I)
master_doc = fitz.open()

for pdf_file in sorted(Path(input_folder).glob("*.pdf")):

    try:
        doc = fitz.open(pdf_file)

        search_limit = min(50, doc.page_count) # Content section is before in the PDF
        matched_page = None
        for page_num in range(search_limit):

            text = doc[page_num].get_text("text")
            matches = len(pattern.findall(text))
            if matches >= 3:
                matched_page = page_num
                break

        if matched_page is not None:

            start_page = master_doc.page_count
            master_doc.insert_pdf(
                doc,
                from_page=matched_page,
                to_page=matched_page
            )

            page = master_doc[start_page]
            page.insert_text(
                (40, 30),
                f"SOURCE: {pdf_file.stem}",
                fontsize=12,
                color=(1, 0, 0)
            )
        else:
            print(f"No match: {pdf_file.name}")

        doc.close()

    except Exception as e:
        print(f"Error: {pdf_file.name} -> {e}")

# master_doc.save(output_pdf)
# master_doc.close()

# AADHARHFC
# AKUMS
# ALKYLAMINE
# APLLTD
# GHCL
# GODREJAGRO
# ICICIGI
# ICICIPRULI
# IFBIND
# IIFL
# INDUSTOWER
# JINDALSAW
# JKTYRE
# KIRLOSENG
# KIRLPNU
# LT
# LUPIN
# MAPMYINDIA
# MARICO
# METROPOLIS
# OFSS
# PRICOLLTD
# SPLPETRO
# SRF
# TIMKEN
# TMCV
# TMPV
# VOLTAMP
# WABAG


In [ ]:
"Corporate Information","CORPORATE OVERVIEW","STATUTORY REPORTS","FINANCIAL STATEMENTS","Contents","Corporate Governance","Business Responsibility","Sustainability Report","Standalone","Consolidated"

In [ ]:
import re
import fitz

NUM_RE = re.compile(r"\b\(?\d[\d,]*(?:\.\d+)?%?\)?\b")  # 1.23, 1,234.45, (123) etc
ALPHA_RE = re.compile(r"[A-Za-z]+")  # pure words
input_folder = r"C:\Users\kaustubh.keny\Downloads\FINANCE\FULL2"

from collections import Counter

def text_dir(page, max_lines=50):
    """
    (1,0)   = NORMAL
    (0,-1)  = 90° clockwise
    (-1,0)  = 180°
    (0,1)   = 90° counter-clockwise
    """

    dirs = Counter()
    line_count = 0

    for block in page.get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            dirs[tuple(map(round, line["dir"]))] += 1
            line_count += 1
            if line_count >= max_lines:
                break
        if line_count >= max_lines:
            break
    if not dirs:
        return None

    p_dir = dirs.most_common(1)[0][0]
    if p_dir == (0, -1):
        return "90_CCLK"
    elif p_dir == (0, 1):
        return "90_CLK"
    elif p_dir == (-1, 0):
        return "UPSIDE_DOWN"

    return "NORMAL"
    
def detect_spread(page, ref_width, ref_height):

    width = page.rect.width
    height = page.rect.height
    w_ratio = width / ref_width
    h_ratio = height / ref_height

    is_spread = w_ratio > 1.6 and h_ratio > 0.7

    return {
        "width": width,
        "height": height,
        "width_ratio": w_ratio,
        "height_ratio": h_ratio,
        "area_ratio": (width * height) / (ref_width * ref_height),
        "spread": is_spread,
    }

def clean_pdf_text(text):

    text = re.sub(r"[\t\r\n]+", " ", text)
    text = re.sub(r"\b(?:2024|2025)\b", " ", text) #2025, 2024 are false positives
    text = re.sub(r"[^A-Za-z0-9\s.,()%]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

def tabular_distribution(page, rect):

    y0 = rect.y0 + rect.height * 0.20
    y1 = rect.y1 - rect.height * 0.20

    split_x = rect.x0 + rect.width * 0.55

    left_rect = fitz.Rect(rect.x0, y0, split_x, y1)
    right_rect = fitz.Rect(split_x, y0, rect.x1, y1)

    # left_text = clean_pdf_text(page.get_text("text", clip=left_rect))
    # right_text = clean_pdf_text(page.get_text("text", clip=right_rect))
    left_words = []
    right_words = []
    words = page.get_text("words")
    for w in words:
        x0, y0, x1, y1, txt = w[:5]

        center_x = (x0 + x1) / 2

        if center_x < split_x:
            left_words.append(txt)
        else:
            right_words.append(txt)

    left_text = " ".join(left_words)
    right_text = " ".join(right_words)    
    
    nl = NUM_RE.findall(left_text)
    nr = NUM_RE.findall(right_text)
    return {
        "alpha_left": len(ALPHA_RE.findall(left_text)),
        "alpha_right": len(ALPHA_RE.findall(right_text)),
        "num_left": len(nl),
        "num_right": len(nr),
        "nl_count":"|".join(nl),
        "nr_count":"|".join(nr)
    }

all_content = []
pdf_content = []
for pdf_file in sorted(Path(input_folder).glob("*.pdf")):

    try:
        
        doc = fitz.open(pdf_file)
        total_pages = doc.page_count

        page_0 = doc[0]
        ref_height = page_0.rect.height
        ref_width = page_0.rect.width
        
        start_t = time.perf_counter()
        
        for i in range(0, total_pages):
            page = doc[i]
            dir_val = text_dir(page)
            spread_info = detect_spread(page, ref_width, ref_height)
            result = {"text_dir":text_dir}
            if not spread_info["spread"]:
                result = {"pdf_name": pdf_file.stem, "page": i + 1, "segment": "FULL", "text_dir":dir_val}

                result.update(spread_info)
                result.update(tabular_distribution(page, page.rect))

                all_content.append(result)

            else:

                rect = page.rect
                mid_x = rect.x0 + rect.width / 2

                left_page = fitz.Rect(rect.x0, rect.y0, mid_x, rect.y1)
                right_page = fitz.Rect(mid_x, rect.y0, rect.x1, rect.y1)
                left_result = {
                    "pdf_name": pdf_file.stem,
                    "page": i + 1,
                    "segment": "LEFT_PAGE", "text_dir":dir_val
                }

                left_result.update(spread_info)
                left_result.update(tabular_distribution(page, left_page))
                all_content.append(left_result)

                right_result = {
                    "pdf_name": pdf_file.stem,
                    "page": i + 1,
                    "segment": "RIGHT_PAGE", "text_dir":dir_val
                }

                right_result.update(spread_info)
                right_result.update(tabular_distribution(page, right_page))
                all_content.append(right_result)

        doc.close()


    except Exception as e:
        print(f"Error: {pdf_file.name} -> {e}")
    end_t = time.perf_counter()
    pdf_content.append({
        "pdf_name":pdf_file.stem,
        "total_pages":total_pages,
        "size_mb": os.path.getsize(str(pdf_file)) / (1024 * 1024),
        # "start_t":start_t,
        # "end_t":end_t,
        "time_taken": end_t - start_t,
        "pages_per_sec": total_pages / max(end_t - start_t, 1e-9)
    })

df = pd.DataFrame(all_content)
df2 = pd.DataFrame(pdf_content)

In [35]:
DEBUG_PDF = "2025_CRAFTSMAN"
def draw_debug_boxes(page):
    rect = page.rect

    y0 = rect.y0 + rect.height * 0.20
    y1 = rect.y1 - rect.height * 0.20
    split_x = rect.x0 + rect.width * 0.55

    left_rect = fitz.Rect(rect.x0, y0, split_x, y1)
    right_rect = fitz.Rect(split_x, y0, rect.x1, y1)

    page.draw_rect(
        left_rect,
        color=(0, 0, 1),      # border = blue
        fill=(0, 0, 1),       # fill = blue
        fill_opacity=0.15,    # 15% opacity
        width=2
    )

    page.draw_rect(
        right_rect,
        color=(0, 1, 0),      # border = green
        fill=(0, 1, 0),       # fill = green
        fill_opacity=0.15,
        width=2
    )

    # 60% divider (blue)
    # page.draw_line(
    #     fitz.Point(split_x, rect.y0),
    #     fitz.Point(split_x, rect.y1),
    #     color=(0, 0, 1),
    #     width=2
    # )

    # spread midpoint (red)
    # mid_x = rect.x0 + rect.width / 2
    # page.draw_line(
    #     fitz.Point(mid_x, rect.y0),
    #     fitz.Point(mid_x, rect.y1),
    #     color=(1, 0, 0),
    #     width=2
    # )

    # top/bottom cutoffs
    page.draw_line(
        fitz.Point(rect.x0, y0),
        fitz.Point(rect.x1, y0),
        color=(0, 0, 1),
        width=2
    )

    page.draw_line(
        fitz.Point(rect.x0, y1),
        fitz.Point(rect.x1, y1),
        color=(0, 0, 1),
        width=2
    )

for pdf_file in sorted(Path(input_folder).glob("*.pdf")):

    doc = fitz.open(pdf_file)
    total_pages = doc.page_count
    for i in range(total_pages):
        page = doc[i]
        
        if pdf_file.stem == DEBUG_PDF:
            draw_debug_boxes(page)

    # save annotated copy
    if pdf_file.stem == DEBUG_PDF:
        doc.save(pdf_file.with_stem(pdf_file.stem + "_DEBUG"))

    doc.close()

In [33]:
# df = pd.DataFrame(all_content)
df["total_alpha"] = df["alpha_left"] + df["alpha_right"]
df["total_num"] = df["num_left"] + df["num_right"]

df["left_alpha_ratio"] = np.where(
    df["total_alpha"] > 0,
    round(100 * df["alpha_left"] / df["total_alpha"],2),
    np.nan
)

df["right_alpha_ratio"] = np.where(
    df["total_alpha"] > 0,
    round(100 * df["alpha_right"] / df["total_alpha"],2),
    np.nan
)

df["left_num_ratio"] = np.where(
    df["total_num"] > 0,
    round(100 * df["num_left"] / df["total_num"],2),
    np.nan
)

df["right_num_ratio"] = np.where(
    df["total_num"] > 0,
    round(100 * df["num_right"] / df["total_num"],2),
    np.nan
)

df["alpha_dominance"] = np.select(
    [
        df["total_alpha"] == 0,
        df["left_alpha_ratio"] > df["right_alpha_ratio"],
        df["left_alpha_ratio"] < df["right_alpha_ratio"]
    ],
    [
        None,
        "LEFT",
        "RIGHT"
    ],
    default="EQUAL"
)

df["num_dominance"] = np.select(
    [
        df["total_num"] == 0,
        df["left_num_ratio"] > df["right_num_ratio"],
        df["left_num_ratio"] < df["right_num_ratio"]
    ],
    [
        None,
        "LEFT",
        "RIGHT"
    ],
    default="EQUAL"
)

df["%_alpha"] = abs(round(df["left_alpha_ratio"] - df["right_alpha_ratio"],2))
df["%_num"] = abs(round(df["left_num_ratio"] - df["right_num_ratio"],2))
# df.to_csv("NUMERIC_RATIO_FULL.csv", index = False)

In [34]:
import pandas as pd
with pd.ExcelWriter("NUMERIC_RATIO_FULL2.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Summary", index=False)
    df2.to_excel(writer, sheet_name="Details", index=False)

In [26]:
cond1 = df["pdf_name"] == "2025_ADANIENT"
cond2 = (df["alpha_dominance"] == "LEFT") & (df["num_dominance"] == "RIGHT")

imp_df = df[cond1 & cond2]


In [ ]:
pages = imp_df.page.to_list()
pages

In [29]:
#get_pdf
input_pdf = r"C:\Users\kaustubh.keny\Projects\INPUTS\ANNUAL_REPORTS\ANNUAL_REPORTS_2025\2025_ADANIENT.pdf"
output_pdf = "selected_pages.pdf"

# pages = [1, 3, 5, 8, 10]

src = fitz.open(input_pdf)
dst = fitz.open()

for p in pages:
    dst.insert_pdf(src, from_page=p - 1, to_page=p - 1)

dst.save(output_pdf)
dst.close()
src.close()

In [ ]:
input_pdf = r"C:\Users\kaustubh.keny\Projects\INPUTS\ANNUAL_REPORTS\ANNUAL_REPORTS_2025\2025_ADANIENT.pdf"

doc = fitz.open(input_pdf)
page = doc[545]

data = page.get_text("dict")

for block in data["blocks"]:
    for line in block.get("lines", []):
        print(line["dir"])